In [112]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import time
import calendar

In [113]:
# Set up WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run in background (remove if debugging)
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [114]:
# Open the insurance page
driver.get("https://www.income.com.sg/buy/travel-insurance")

# Wait for form elements to load
wait = WebDriverWait(driver, 10)
time.sleep(3)

In [115]:
def screenshot(phase, y=0):
    driver.execute_script(f"window.scrollBy(0, {y});")
    time.sleep(3)

    driver.save_screenshot(f"after_click_{phase}.png")
    print(f"========== {phase} DONE ==========")

In [116]:
# Select policy type
policy_type = "per-trip"

if (policy_type == "per-trip"):
    policy_id = "evgTripPolicy"
elif (policy_type == "yearly"):
    policy_id = "evgYearlyPolicy"
else:
    raise Exception("Invalid policy type: " + policy_type)
policy_selector = wait.until(EC.element_to_be_clickable((By.ID, policy_id)))
policy_selector.click()

# TODO: haven't done yearly

screenshot("policy", 600)

========== policy DONE ==========


In [117]:
# Select coverage type
coverage_type = "individual"

if (coverage_type == "individual"):
    coverage_id = "Individual/Group"
elif (coverage_type == "family"):
    coverage_id = "Family"
else:
    raise Exception("Invalid coverage type: " + coverage_type)
coverage_selector = wait.until(EC.element_to_be_clickable((By.XPATH, f"//li[@data-target='coverageType' and @data-value='{coverage_id}']")))
driver.execute_script("arguments[0].click();", coverage_selector)

# TODO: haven't done family

screenshot("coverage", 200)

========== coverage DONE ==========


In [118]:
# Select destination
destination = 'asia'

destination_selector = wait.until(EC.element_to_be_clickable((By.XPATH, "//input[contains(@placeholder, 'Select destination(s)')]")))
destination_selector.click()

screenshot('destination_1')

need_scroll = False

if (destination == 'one-or-more'):
    region = 'One or more countries'
elif (destination == 'asean'):
    region = 'ASEAN'
elif (destination == 'asia'):
    region = 'Asia'
    need_scroll = True
elif (destination == 'worldwide'):
    region = 'Worldwide'
    need_scroll = True
else:
    raise Exception("Invalid destination: " + destination)

region_input = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, f"label[data-value='{region}']")))
region_input.click()

if (destination == 'asean' or destination == 'asia' or destination == 'worldwide'):
    understand_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'I UNDERSTAND')]")))
    understand_button.click()

# TODO: one-or-more not done

screenshot('destination_2')

========== destination_1 DONE ==========
========== destination_2 DONE ==========


In [119]:
def date_splitter(date):
    day, month, year = date.split('/')
    month_name = calendar.month_name[int(month)]  # e.g., 5 -> "May"
    return day, month_name, year

def click_calendar(date):
    target_date, target_month, target_year = date_splitter(date)

    wait = WebDriverWait(driver, 10)
    calendar_display = wait.until(EC.visibility_of_element_located((By.ID, 'ui-datepicker-div')))

    # find correct month & year
    while True:
        displayed_month = calendar_display.find_element(By.CLASS_NAME, "ui-datepicker-month").text
        displayed_year = calendar_display.find_element(By.CLASS_NAME, "ui-datepicker-year-no-dropdown").text

        if displayed_month == target_month and displayed_year == target_year:
            break

        next_button = driver.find_element(By.CLASS_NAME, "ui-datepicker-next")
        next_button.click()
        time.sleep(0.5)

    # find correct date
    date_element = calendar_display.find_element(By.XPATH, f".//a[@data-date='{target_date}']")
    date_element.click()

In [120]:
# Input departure date
departure_date = '22/05/2025'

date_input = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-test="c_startDate"]')))
date_input.click()

screenshot("departure_date_1")

click_calendar(departure_date)

screenshot("departure_date_2")

========== departure_date_1 DONE ==========
========== departure_date_2 DONE ==========


In [121]:
# Input arriving date
arriving_date = '22/06/2025'

date_input = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-test="c_endDate"]')))
date_input.click()

screenshot("arriving_date_1")

click_calendar(arriving_date)

screenshot("arriving_date_2")

========== arriving_date_1 DONE ==========
========== arriving_date_2 DONE ==========


In [122]:
# # Select left Singapore option
# left_singapore = True

# if left_singapore:
#     yes_btn = driver.find_element(By.ID, "hasDeparted")
#     yes_btn.click()
# else:
#     no_btn = driver.find_element(By.ID, "notDeparted")
#     no_btn.click()

# screenshot("left_singapore")

In [123]:
# Click consent checkbox
consent_checkbox = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[@class='checkmark']")))
driver.execute_script("arguments[0].click();", consent_checkbox)

screenshot("consent_box", 1000)

========== consent_box DONE ==========


In [124]:
# Click "Get Quote" button
get_quote_button = driver.find_element(By.CSS_SELECTOR, '[data-test="b_quote"]')
get_quote_button.click()

wait.until(EC.presence_of_element_located((By.CLASS_NAME, "per-trip-policy-plans")))

screenshot("quote")

========== quote DONE ==========


In [125]:
plan_data = []

# Locate all plan columns
plan_columns = driver.find_elements(By.CLASS_NAME, "plan-item-header")

for plan in plan_columns:
    plan_info = {}

    # Get plan name
    plan_name_element = plan.find_element(By.CLASS_NAME, "plan-name")
    plan_name_text = plan_name_element.text.strip().split("\n")[0]

    if not plan_name_text:
        continue

    plan_info["plan_name"] = plan_name_text

    # Get prices
    price_sections = plan.find_elements(By.CLASS_NAME, "th-2col")

    if len(price_sections) >= 2:
        # Adult price
        adult_price = price_sections[0].find_element(By.CLASS_NAME, "th-item-left").find_elements(By.TAG_NAME, "div")[1].text.strip()
        plan_info["price_per_adult"] = adult_price

        # Child price
        child_price = price_sections[1].find_element(By.CLASS_NAME, "th-item-left").find_elements(By.TAG_NAME, "div")[1].text.strip()
        plan_info["price_per_child"] = child_price
    else:
        # In case child price is not available
        plan_info["price_per_adult"] = "N/A"
        plan_info["price_per_child"] = "N/A"

    plan_data.append(plan_info)

print(plan_data)

# Display the result
for plan in plan_data:
    print(plan)


[{'plan_name': 'Classic', 'price_per_adult': '$269.34', 'price_per_child': '$215.00'}, {'plan_name': 'Deluxe', 'price_per_adult': '$311.70', 'price_per_child': '$242.52'}, {'plan_name': 'Preferred', 'price_per_adult': '$407.10', 'price_per_child': '$304.52'}]
{'plan_name': 'Classic', 'price_per_adult': '$269.34', 'price_per_child': '$215.00'}
{'plan_name': 'Deluxe', 'price_per_adult': '$311.70', 'price_per_child': '$242.52'}
{'plan_name': 'Preferred', 'price_per_adult': '$407.10', 'price_per_child': '$304.52'}
